# CopyLane 판정 인코더 — D4-15 재분할 재학습 : D4-10 + 해설서 재분할 학습 950행 (박수진, 2026-10-15)

> 골든 **`f6ccc5a7de44`** · 재분할 **v1** (D-347 · 팀장 10-15 · `preprocess/encoder_resplit.py`) — 학습 950행 `e8f5d6d27e7a5cea` · dev 398행 `cc2d8b7e42d57266`
> dev = **확장 dev 1,147행** (공통 dev 749행 `4ebfcb231da703b5` + 재분할 dev 398행) — 회차 선택 · 문턱 · 보고 전부
> 등록 `docs/psj/reports/D4_실험기록_20261012.md` **D4-15** (돌리기 전에 올렸다) · 🔴 **dev 만 본다** — 새 봉인 평가셋 1,141행은 메모리에서 바로 버린다 (D-347 · D-175)

## 무엇을 하나

D4-10 노트북을 그대로 쓰고 **학습 입력에 재분할 학습 행을 더한다.** 비교를 위해 같은 확장 dev 로 D4-10 레시피도 다시 학습한다.

| 실험 | 학습 | 쓰임 |
|---|---|---|
| `D4-15-재분할학습` | D4-10 학습 4,511행 + 재분할 학습 950행 (D4-10 과 같은 규칙: 조건 M · 유형 없는 위반 제외 · dev 와 같은 문구 제거 · 같은 문구 병합) | **배포 판 후보** — 가장 좋은 시드의 모델 zip |
| `D4-10재현-확장dev` | D4-10 학습 4,511행 그대로 | 같은 확장 dev 에서의 비교 기준 (모델 저장 안 함) |

- 입력 · 손실 · 설정은 D4-10 그대로 — 문장쌍 [품목 · 문장] · 136 토큰 · 학습 때 50% 「모름」 · Focal γ 1.5 · lr 2e-5 · 6 epoch · 시드 48 · 49 · 50.
- 회차는 확장 dev 의 (가) 품목 입력 6종 PR-AUC 로 고른다. D-324 문턱 · 판정은 로컬 `d4_15_resplit.py` (목표 재현율 = D4-15a 노트북이 다시 잰 카드 17 기준선 재현).

## 실행 순서

1. **Colab** — 런타임 GPU → `golden.jsonl` · 정상 문구 CSV · `resplit_v1_train.txt` · `resplit_v1_dev.txt` 를 올리라고 묻는다. 결과는 Drive `copylane_D4-15/`.
   **RunPod** — 네 파일을 노트북 옆에 둔다. 결과는 `./copylane_D4-15/`.
2. 학습은 2 실험 × 시드 3 = 6번 (T4 약 25분). 끊겨도 다시 실행하면 끝난 시드는 건너뛴다.
3. 맨 아래 셀이 **`D4-15_csv.zip`** 을 만든다. 모델 zip `copylane-encoder-kcbert-D4-15-재분할학습.zip` 은 같은 폴더.


## 1. 환경 설정 + 파일 업로드 (골든 · 정상 합성 문구 CSV)

`sympy` 충돌로 `from transformers import ...` 가 깨지면 런타임 > 세션 다시 시작 후 첫 셀은 건너뛰고 다음 셀부터 실행한다.

In [ ]:
!pip install -q -U sympy transformers accelerate datasets scikit-learn pandas

In [ ]:
import os, json, hashlib, shutil, gc, math, random, csv, time, zipfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import files as colab_files          # Colab — 없으면 묻는다
except ImportError:
    colab_files = None                                     # RunPod · 로컬 — 두 파일을 이 노트북 옆에 미리 둔다
if not os.path.exists("golden.jsonl"):
    if colab_files is None:
        raise SystemExit("🔴 golden.jsonl 이 없다 — 이 노트북과 같은 폴더에 둔다 (RunPod: VS Code 탐색기로 끌어 놓기)")
    print("① golden.jsonl 선택")
    colab_files.upload()
SYN_NAME = "LLM_카피생성_정상카피_900_판독용_사람칸채움_ksr_2026-10-11.csv"     # 🆕 카드 17 — 권소라 검수판 (판독(사람) 칸)
if not os.path.exists(SYN_NAME):
    if colab_files is None:
        raise SystemExit(f"🔴 {SYN_NAME} 이 없다 — 이 노트북과 같은 폴더에 둔다")
    print("② 정상 합성 문구 CSV 선택 —", SYN_NAME)
    for _f in colab_files.upload():                  # 브라우저가 이름을 바꿔 올렸으면(「 (1)」 등) 제 이름으로 둔다
        if _f.endswith(".csv") and _f != SYN_NAME:
            os.replace(_f, SYN_NAME)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GOLDEN_SHA = sha256("golden.jsonl")
print("golden.jsonl sha256:", GOLDEN_SHA[:16], "…")
# D-176: 입력 지문을 **대조**한다
EXPECTED_GOLDEN_SHA = "f6ccc5a7de44c6863ebb901d4e5a726acb35c92527e41fd1c3850490158fe744"   # 🆕 10-10 2 차 재동결 (8,383행 · `전제` 칸)
EXPECTED_ROWS = (8383, 5894, 2489)   # 전체 · 학습 · 평가
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176). "
                     "런타임에 예전 파일이 남았으면 런타임을 새로 시작한다")
SYN_SHA = sha256(SYN_NAME)
EXPECTED_SYN_SHA = "f3bfec9079e05c367b2d27495c385659118fd56abd963e89ac5d760abfdd00ea"      # 🆕 권소라 10-11 검수판 (900행 · 판독(사람) 칸)
if SYN_SHA != EXPECTED_SYN_SHA:
    raise SystemExit(f"🔴 정상 합성 문구 CSV 가 카드에 적은 판과 다르다 — 받은 {SYN_SHA[:16]} · 기대 {EXPECTED_SYN_SHA[:16]}. "
                     "판독 칸을 채운 새 판이면 새 카드로 돌린다")
print("정상 합성 문구 CSV sha256:", SYN_SHA[:16], "…")
# 🆕 D4-15 (D-347) — 재분할 v1 id 목록. 로컬에서 `uv run python -m preprocess.encoder_resplit --write build/resplit` 로 만든 파일 (id 만 · 글자 없음)
RESPLIT_KINDS = ['train', 'dev']
RESPLIT_EXPECT = {"train": (950, "e8f5d6d27e7a5cea"), "dev": (398, "cc2d8b7e42d57266")}     # 팀장 전달 10-15 · D-347
_need = [f"resplit_v1_{k}.txt" for k in RESPLIT_KINDS if not os.path.exists(f"resplit_v1_{k}.txt")]
if _need:
    if colab_files is None:
        raise SystemExit(f"🔴 {_need} 이 없다 — 이 노트북과 같은 폴더에 둔다 (build\\resplit\\)")
    print("③ 재분할 목록 선택 —", "resplit_v1_train.txt, resplit_v1_dev.txt", "(한 번에 골라도 된다)")
    for _f in colab_files.upload():
        for k in RESPLIT_KINDS:
            if _f.endswith(".txt") and f"_{k}" in _f and _f != f"resplit_v1_{k}.txt":
                os.replace(_f, f"resplit_v1_{k}.txt")
RESPLIT_IDS = {}
for k in RESPLIT_KINDS:
    _ids = [l.strip() for l in open(f"resplit_v1_{k}.txt", encoding="utf-8") if l.strip()]
    _mark = hashlib.sha256("|".join(sorted(_ids)).encode()).hexdigest()[:16]      # preprocess.devsplit.sha 와 같은 식
    if (len(_ids), _mark) != RESPLIT_EXPECT[k] or len(set(_ids)) != len(_ids):
        raise SystemExit(f"🔴 resplit_v1_{k}.txt 가 D-347 과 다르다 — {len(_ids)}행 · 지문 {_mark} · 기대 {RESPLIT_EXPECT[k]}")
    RESPLIT_IDS[k] = _ids
    print(f"✅ 재분할 {k} {len(_ids)}행 · 지문 {_mark}")


## 2. 스위치 — 바꿀 것은 이 셀뿐이다

| 스위치 | 기본값 | 뜻 |
|---|---|---|
| `SAVE_MODEL_ZIP` | `False` | 탐색이라 기본은 저장하지 않는다. 통과하고 팀장이 열자고 하면 `True` 로 다시 돌린다 |
| `SMOKE` | `False` | `True` = 1분짜리 점검(시드 1 · 1 epoch · 학습 300줄). 수치는 의미 없다 |
| `CAT_DROP` · `MAX_LEN` | 0.5 · 136 | 🔒 **D4 기록에 적은 값이다. 고치지 않는다** |
| `SYN_HOLD_PER_CAT` · `SYN_SEED` | 5 · 42 | 🔒 정상 문구를 나누는 방법 — 카드 17 과 같다 |


In [ ]:
RUN_DEFS = {"D4-15-재분할학습": {"guide": True},      # 🆕 D4-15 — D4-10 + 재분할 학습 950행 (D-347) · 배포 판 후보
            "D4-10재현-확장dev": {"guide": False}}    # D4-10 레시피 그대로 · 같은 확장 dev — 비교 기준 (모델 저장 안 함)
RUNS = ["D4-15-재분할학습", "D4-10재현-확장dev"]   # 끊길 때를 생각해 후보를 먼저 돌린다
RUN = "D4-15-재분할학습"
SAVE_MODEL_ZIP = True              # RUN 의 가장 좋은 시드 모델만 남긴다
SMOKE = False

# ── 🔒 입력 — D4-04 와 같다 (D-336)
CAT_VALUES = ["식품", "건기식", "화장품"]     # 골든 `품목` 칸 · 정상 문구 CSV `품목` 칸의 값. 그 밖 · 빈 칸은 「모름」
CAT_UNKNOWN = "모름"
CAT_DROP = 0.50                              # 학습 때 품목이 있는 행을 「모름」으로 바꿔 넣는 비율 (배치마다 새로 뽑는다 · 옛 카드 6 은 0.30)
MAX_LEN = 136                                # [CLS] 품목 [SEP] 문장 [SEP] 전체 — D4-00 은 문장만 128
MODES = ["품목", "모름"]                     # (가) 품목을 넣는다 / (나) 전부 「모름」

# ── 학습 설정 — D4-00 · D4-04 와 같다. 바꾸지 않는다
MODEL_NAME = "beomi/kcbert-base"
SEEDS = [48, 49, 50]                # D4-00 과 같은 시드 — 짝짓는다
MAX_EPOCHS, PATIENCE = 6, 2
BATCH, LR, WARMUP = 32, 2e-5, 0.1
WEIGHT_DECAY = 0.01
FOCAL_GAMMA, POS_WEIGHT_CAP = 1.5, 5
USE_WORD_ROWS = False             # 사례집 낱말 행 제외
DEV_FRAC, DEV_SEED = 0.15, 42     # 공통 dev — 바꾸면 dev 행이 달라진다
TH_OFF = 1.01                     # 편입 대기 칸(기능성화장품_오인)의 문턱 — 꺼 둠 (D-321)
SCORE_VIEW = "v11"                # 🆕 D4-15 — 회차 · 시드는 확장 dev 1,147행 전부로 고른다 — 합성 정상 dev 로는 아무것도 고르지 않는다

# ── 🔒 정상 합성 문구를 나누는 방법 — 카드 17 과 같다
SYN_HOLD_PER_CAT, SYN_SEED = 5, 42
SYN_REVIEW_OK = "적법"            # 🔒 D-325
SYN_FACT_OK = "적법(사실 전제)"   # 🆕 D4-10 — 학습 음성으로 더하는 검수 낱말 (D-331 결정 7)
SYN_EXPECT = {"학습": 334, "학습-사실 전제": 211, "합성 정상 dev": 171, "참고(사실 전제 dev · 애매 · 조건부)": 184}      # 로컬 대조 10-13
QUIET_MARGIN = 0.5                # 판정 로직의 여유 구간 (미리 보기용)

assert 0 <= CAT_DROP < 1 and CAT_UNKNOWN not in CAT_VALUES
if SMOKE:
    SEEDS, MAX_EPOCHS = [48], 1
NOTEBOOK = "D4-15"
def run_tag(name):
    return f"copylane-encoder-kcbert-{name}"
TAG = run_tag(RUN)

OUT_ROOT = "/content/drive/MyDrive/copylane_D4-15" + ("_smoke" if SMOKE else "")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(OUT_ROOT, exist_ok=True)
except Exception as e:
    OUT_ROOT = "./copylane_D4-15" + ("_smoke" if SMOKE else "")      # RunPod · 로컬 — 노트북 옆 폴더
    os.makedirs(OUT_ROOT, exist_ok=True)
    print("ℹ️ Colab 이 아니다 — 결과를", os.path.abspath(OUT_ROOT), "에 둔다 (파드를 Terminate 하면 사라진다 · 끝나면 D4-15_csv.zip · 모델 zip 을 받는다)")
print("실험:", RUNS, "· 시드", SEEDS, "· 「모름」 비율", CAT_DROP, "· 결과 폴더", OUT_ROOT)


## 3. 데이터 + dev 분리 — 공통 dev (Baseline 노트북과 같은 셀)

1. 묶음 = 같은 원문에서 나온 행 (`inj:T1:hf:0:0` → `hf:0:0` · 그 밖은 `id` 의 `#` 앞).
2. 원천마다 묶음의 15% 를 dev 로 (시드 42 · 묶음 이름순 → 섞기).
3. 6종마다 dev 양성이 10건보다 적으면 남은 묶음에서 채운다.
4. dev 행 = dev 묶음의 **문장 단위 · 합성 아님** 행.

같은 골든이면 **749줄 · 지문 `4ebfcb231da703b5`** 가 나와야 한다 — v11 의 `dev_ids_v11_seed42.csv` 와 같은 행 · 같은 순서다. 다르면 멈춘다.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

SEL = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]   # 모델을 고르는 6종 (D-321 결정 4)
NEW_CONF = ["부당_비교광고", "비방광고"]
CONF8 = SEL + NEW_CONF                       # 확정 8종 — 문턱을 dev 에서 고른다
WAITING = ["기능성화장품_오인"]              # 편입 대기 — 칸은 두고 문턱은 끈다
LABEL_LIST = CONF8 + WAITING                 # 저장 순서 (팀장 10-06)
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
N_SEL, N8, N_ALL = len(SEL), len(CONF8), len(LABEL_LIST)

def norm_text(s):
    return " ".join(s.split())

def group_key(rid):
    if rid.startswith("inj:"):               # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def encoder_target(r):
    """인코더의 표적 — `docs/psj/e2e_prototype/score_encoder_dev.py` 의 `encoder_target` 과 같다 (🆕 카드 17).
    조건 M · D · L 은 음성 (D-296 개정 2) · 전제 `건기식_비인정` 행은 `거짓_과장` → `건강기능식품_오인` (D-326 보충 12) ·
    근거에 4 호 나목 말고 다른 4 호 목이 있으면 `거짓_과장` 도 남긴다. 🚨 골든 labels 는 고치지 않는다."""
    if r.get("조건") in ("M", "D", "L"):
        return []
    t = list(r.get("labels") or [])
    if r.get("전제") == "건기식_비인정" and "거짓_과장" in t:
        other4 = any("제4호" in g and "|나목" not in g for g in (r.get("근거") or []))
        t = [x for x in t if x != "거짓_과장" or other4]
        if "건강기능식품_오인" not in t:
            t.append("건강기능식품_오인")
    return t

def normalize(r):
    cond, labels = r.get("조건"), list(r.get("labels") or [])
    return {"id": r["id"], "group": group_key(r["id"]), "text": r["text"], "labels": labels,
            # target 이 학습 · 채점의 정답이다 — 조건 M · D · L 음성 · 전제 건기식_비인정 표적 변경 (encoder_target)
            "target": encoder_target(r), "premise": r.get("전제"),
            "split": r["split"], "provenance": r.get("provenance"), "origin": r.get("origin"),
            "zone": r.get("구역"), "cond": cond, "unit": r.get("unit"),
            "cat": r.get("품목")}     # 🆕 D4-04 — 품목 칸을 읽는다

def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"

def is_inj(r):
    return r["origin"] == "injected" or r["id"].startswith("inj:")

def bucket(r):
    # 8종 기준 (run_judge_dist.py 와 같은 갈래)
    if r["cond"] == "M": return "M"
    if r["cond"] == "D": return "D"
    if r["cond"] == "L": return "L"
    if any(l in CONF8 for l in r["labels"]): return "scored"
    if r["labels"]: return "대기"
    return "pending" if r["cond"] in ("C", "A", "B") else "neg"

golden = load_jsonl("golden.jsonl")
all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(LABEL_LIST)
assert not unknown, f"🔴 예상 밖 라벨: {unknown}"
train_all = [r for r in all_rows if r["split"] == "train"]
_got = (len(all_rows), len(train_all), len(all_rows) - len(train_all))
if _got != EXPECTED_ROWS:
    raise SystemExit(f"🔴 행 수가 다르다 — 받은 {_got} · 기대 {EXPECTED_ROWS} (D-176)")
# 🆕 D4-15 (D-347) — 봉인 평가셋에서 인코더로 옮긴 해설서 행만 꺼낸다. 목록에 없는 test 행(새 봉인 1,141행)은 아래 del 로 버린다 — 열지 않는다
_by_id = {r["id"]: r for r in all_rows}
_ids_all = [i for k in RESPLIT_IDS for i in RESPLIT_IDS[k]]
if any(i not in _by_id for i in _ids_all) or any(_by_id[i]["split"] != "test_sentence" for i in _ids_all) or len(set(_ids_all)) != len(_ids_all):
    raise SystemExit("🔴 재분할 목록이 이 골든의 봉인 행과 맞지 않는다 (D-347)")
GUIDE = {k: [_by_id[i] for i in RESPLIT_IDS[k]] for k in RESPLIT_IDS}
assert all(r["unit"] == "문장" and r["origin"] != "injected" for k in GUIDE for r in GUIDE[k]), "재분할 행에 낱말 · 주입 행이 있다"
del _by_id, _ids_all
del golden, all_rows      # 🔴 여기서부터 test 행은 메모리에도 없다 — 이 노트북은 train(→ train · dev)만 본다

# ══ dev 분리 — v11 (encoder_finetune_colab_v11_kcbert_1006) 섹션 3 과 같은 코드
def dev_eligible(r):
    return r["unit"] == "문장" and r["origin"] != "injected"

groups = defaultdict(list)
for r in train_all:
    groups[r["group"]].append(r)

prov_groups = defaultdict(list)             # 원천별 15% (묶음 단위 · 시드 고정)
for g in sorted(groups):
    elig = [r for r in groups[g] if dev_eligible(r)]
    if elig:
        prov_groups[Counter(r["provenance"] for r in elig).most_common(1)[0][0]].append(g)
rng = random.Random(DEV_SEED)
dev_groups = set()
for prov in sorted(prov_groups):
    gs = prov_groups[prov][:]
    rng.shuffle(gs)
    dev_groups.update(gs[:max(1, math.ceil(DEV_FRAC * len(gs)))])

def conf_pos(rows, l):
    return sum(1 for r in rows if dev_eligible(r) and l in r["target"])
rest = [g for g in sorted(groups) if g not in dev_groups]
rng.shuffle(rest)
for l in SEL:                               # 6종마다 dev 양성이 10건보다 적으면 채운다
    total = conf_pos(train_all, l)
    want = min(10, math.ceil(DEV_FRAC * total))
    have = sum(conf_pos(groups[g], l) for g in dev_groups)
    for g in rest:
        if have >= want:
            break
        k = conf_pos(groups[g], l)
        if k and g not in dev_groups:
            dev_groups.add(g); have += k

dev_rows = [r for g in sorted(dev_groups) for r in groups[g] if dev_eligible(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
DEV_SHA = hashlib.sha256("|".join(sorted(r["id"] for r in dev_rows)).encode()).hexdigest()[:16]
DEV_EXPECT = {"529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe": (749, "4ebfcb231da703b5"),
              "f6ccc5a7de44c6863ebb901d4e5a726acb35c92527e41fd1c3850490158fe744": (749, "4ebfcb231da703b5")}   # 🆕 10-10 판 — dev 행은 같다   # 골든 sha → v11 dev (줄 수 · 지문)
if GOLDEN_SHA in DEV_EXPECT:
    if (len(dev_rows), DEV_SHA) != DEV_EXPECT[GOLDEN_SHA]:
        raise SystemExit(f"🔴 dev 가 v11 과 다르다 — {len(dev_rows)}줄 · 지문 {DEV_SHA} · 기대 {DEV_EXPECT[GOLDEN_SHA]}. 분리 코드를 건드렸는지 본다")
    print(f"✅ dev = v11 과 같은 행: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
else:
    print(f"🟡 이 골든의 v11 dev 지문을 모른다 — 대조 없이 간다: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
assert len({r["id"] for r in dev_rows}) == len(dev_rows), "dev 에 같은 id 가 둘 이상이다"

# 🆕 D4-15 (D-347) — 확장 dev = 공통 dev 749 + 재분할 dev 398. DEV_SHA(label_scheme dev_sha)는 공통 dev 것 그대로
DEV749 = list(dev_rows)
assert not ({r["id"] for r in GUIDE["dev"]} & {r["id"] for r in DEV749}), "재분할 dev 가 공통 dev 와 id 가 겹친다"
assert not ({norm_text(r["text"]) for r in GUIDE["dev"]} & dev_texts), "재분할 dev 가 공통 dev 와 문구가 겹친다"
dev_rows = DEV749 + GUIDE["dev"]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
EXT_DEV_SHA = hashlib.sha256("|".join(sorted(r["id"] for r in dev_rows)).encode()).hexdigest()[:16]
print(f"✅ 확장 dev {len(dev_rows)}행 = 공통 {len(DEV749)} + 재분할 {len(GUIDE['dev'])} · 지문 {EXT_DEV_SHA}")

# ── dev 행의 갈래 · 채점 방식
for r in dev_rows:
    r["bucket"] = bucket(r)
B = np.array([r["bucket"] for r in dev_rows])
IS_REASON = np.array([is_reason(r) for r in dev_rows])
VIEW = {"v11": np.ones(len(dev_rows), dtype=bool),
        "A": ~IS_REASON & ~np.isin(B, ["M", "pending"])}
SEL_MASK = VIEW[SCORE_VIEW]
BUCKETS = ["scored", "대기", "pending", "M", "D", "L", "neg"]
BUCKET_KO = {"scored": "유형 있는 위반", "대기": "편입 대기 유형만", "pending": "유형 없는 위반", "M": "조건 M", "D": "조건 D", "L": "조건 L(적법)", "neg": "그 밖 음성"}
print("\ndev 갈래 (전체 / 그중 「이유」 구역):")
for b in BUCKETS:
    if (B == b).any():
        print(f"  {BUCKET_KO[b]:12s} {int((B == b).sum()):4d} / {int(((B == b) & IS_REASON).sum()):4d}")
print(f"  합계           {len(dev_rows):4d} / {int(IS_REASON.sum()):4d}")
print(f"채점 행 — v11 방식 {int(VIEW['v11'].sum())}줄 · A 방식 {int(VIEW['A'].sum())}줄  → 이번 선택 방식: {SCORE_VIEW}")
_moved = [r for r in train_all if r.get("premise") == "건기식_비인정" and r["target"] != ([] if r["cond"] in ("M", "D", "L") else r["labels"])]
print(f"표적을 바꾼 행 (전제 건기식_비인정 · D-326 보충 12): {len(_moved)} — 그중 dev {sum(r['id'] in {d['id'] for d in dev_rows} for r in _moved)}")
if GOLDEN_SHA == EXPECTED_GOLDEN_SHA and not SMOKE:
    assert len(_moved) == 37 and sum(r["id"] in {d["id"] for d in dev_rows} for r in _moved) == 7, "표적 변경 행 수가 팀장 전달(학습 30 · dev 7)과 다르다"

## 4. 학습셋 · 품목 · 정상 합성 문구 나누기 (🆕 사실 전제 211 을 학습에)

골든에서 오는 학습 행을 고르는 규칙은 D4-00 과 같다 — dev 묶음을 뺀 train 에서 dev 와 같은 문구 제거 → 조건 M 제외 → 유형 없는 위반 제외 → 낱말 행 제외.
**다른 곳 하나** — 같은 문구를 합칠 때 **품목이 다르면 따로 둔다**(D4-00 은 품목을 보지 않았다).
정상 문구는 카드 17 과 같은 15개 제품을 뗀다 — 학습: 적법 334 + 사실 전제 211 / 측정: 적법 171 · 사실 전제 110 / 참고: 애매 · 조건부 74. 정상 문구의 품목은 그 CSV 의 `품목` 칸이다.


In [ ]:
base_pool = [r for r in train_all if r["group"] not in dev_groups]
STEPS = [("dev 묶음 뺀 train", len(base_pool))]
pool0 = [r for r in base_pool if norm_text(r["text"]) not in dev_texts];                 STEPS.append(("dev 와 같은 문구 제거", len(pool0)))
pool0 = [r for r in pool0 if r["cond"] != "M"];                                          STEPS.append(("조건 M 제외", len(pool0)))
pool0 = [r for r in pool0 if not (r["cond"] in ("C", "A", "B") and not r["labels"])];    STEPS.append(("유형 없는 위반 제외", len(pool0)))
if not USE_WORD_ROWS:
    pool0 = [r for r in pool0 if r["unit"] != "낱말"];                                   STEPS.append(("낱말 행 제외", len(pool0)))
print(" → ".join(f"{k} {n:,}" for k, n in STEPS))
assert not any(r["cond"] == "M" for r in pool0) and not any(r["target"] for r in pool0 if r["cond"] in ("D", "L"))

def cat_of(r):
    return r["cat"] if r.get("cat") in CAT_VALUES else CAT_UNKNOWN

merged = {}
for r in pool0:                                      # 같은 문구 병합 — 라벨 합집합 · 이유/비이유는 따로 · 🆕 D4-04 품목이 다르면 따로
    k = (cat_of(r), norm_text(r["text"]), is_reason(r))
    if k in merged:
        merged[k]["target"] = sorted(set(merged[k]["target"]) | set(r["target"]))
    else:
        merged[k] = dict(r, target=list(r["target"]))
BASE_TRAIN = list(merged.values())

# ── 정상 합성 문구 — 제품 단위로 나눈다
with open(SYN_NAME, encoding="utf-8-sig", newline="") as f:
    _raw = list(csv.DictReader(f))
assert len(_raw) == 900 and len({r["id"] for r in _raw}) == 900, "정상 합성 문구가 900행이 아니다"
SYN_CATS = ["식품", "건기식", "화장품"]
def syn_row(r):
    return {"id": "syn:" + r["id"], "group": f"syn:{r['품목']}:{r['제품종류']}", "text": r["text"], "labels": [], "target": [],
            "split": "extra", "provenance": "llm_copy_ksr_20261011_reviewed", "origin": "synthetic_lawful", "zone": None, "cond": "정상합성", "unit": "문장",
            "cat": r["품목"], "product": r["제품종류"], "angle": r["각도"], "style": r["문체"],
            # 🆕 카드 17 — 학습 · 목표에는 검수 「적법」 · 애매 표시 없음만. 나머지(애매 · 적법(사실 전제) · 조건부)는 참고로만 잰다 (D-325)
            "review": r["판독(사람)"].strip(), "amb_mark": bool(r["애매표시"].strip()),
            "amb": bool(r["애매표시"].strip()) or r["판독(사람)"].strip() != SYN_REVIEW_OK}
SYN = [syn_row(r) for r in _raw]
assert {r["cat"] for r in SYN} == set(SYN_CATS)
SYN_HOLD = set()
for c in SYN_CATS:
    prods = sorted({r["product"] for r in SYN if r["cat"] == c})
    SYN_HOLD |= {(c, p) for p in random.Random(SYN_SEED).sample(prods, SYN_HOLD_PER_CAT)}
SYN_TRAIN = [r for r in SYN if not r["amb"] and (r["cat"], r["product"]) not in SYN_HOLD]
# 🆕 D4-10 — 「적법(사실 전제)」 · 애매 표시 없음 · 학습 제품 → 학습 음성. 측정 제품의 사실 전제는 갈래 「참고-적법(사실 전제)」로 잰다
SYN_TRAIN_FACT = [r for r in SYN if r["review"] == SYN_FACT_OK and not r["amb_mark"] and (r["cat"], r["product"]) not in SYN_HOLD]
_fact_ids = {r["id"] for r in SYN_TRAIN_FACT}
SYN_AMB = [r for r in SYN if r["amb"] and r["id"] not in _fact_ids]
SYN_DEV = [r for r in SYN if not r["amb"] and (r["cat"], r["product"]) in SYN_HOLD]
SYN_EVAL = SYN_DEV + SYN_AMB                          # 학습에 넣지 않는 행 — 시드마다 확률을 적어 둔다
_got = {"학습": len(SYN_TRAIN), "학습-사실 전제": len(SYN_TRAIN_FACT), "합성 정상 dev": len(SYN_DEV), "참고(사실 전제 dev · 애매 · 조건부)": len(SYN_AMB)}
if _got != SYN_EXPECT:
    raise SystemExit(f"🔴 정상 합성 문구를 나눈 수가 카드와 다르다 — 받은 {_got} · 기대 {SYN_EXPECT}")
_base_texts = {norm_text(r["text"]) for r in BASE_TRAIN}
_dup = [r["id"] for r in SYN if norm_text(r["text"]) in dev_texts or norm_text(r["text"]) in _base_texts]
if _dup:
    raise SystemExit(f"🔴 정상 합성 문구 {len(_dup)}행이 골든의 학습 행 · 공통 dev 와 같은 문구다 (예: {_dup[:3]}) — 음성으로 넣으면 라벨이 부딪친다")
assert not ({r["group"] for r in SYN_TRAIN} & {r["group"] for r in SYN_EVAL if not r["amb"]}), "같은 제품이 학습과 합성 정상 dev 에 함께 있다"
assert not ({norm_text(r["text"]) for r in SYN_TRAIN} & {norm_text(r["text"]) for r in SYN_EVAL}), "학습과 측정에 같은 문구가 있다"
assert not ({r["group"] for r in SYN_TRAIN_FACT} & {r["group"] for r in SYN_EVAL if (r["cat"], r["product"]) in SYN_HOLD}), "같은 제품이 사실 전제 학습과 측정에 함께 있다"
assert not ({norm_text(r["text"]) for r in SYN_TRAIN_FACT} & ({norm_text(r["text"]) for r in SYN_EVAL} | dev_texts)), "사실 전제 학습과 측정 · dev 에 같은 문구가 있다"
print("\n■ 정상 합성 문구 나누기 (품목별)")
display(pd.DataFrame({k: {c: sum(r["cat"] == c for r in rs) for c in SYN_CATS} | {"합계": len(rs)}
                      for k, rs in (("학습", SYN_TRAIN), ("학습-사실 전제", SYN_TRAIN_FACT), ("합성 정상 dev", SYN_DEV), ("참고(사실 전제 dev · 애매 · 조건부)", SYN_AMB))}).T)
print("검수 낱말 (판독(사람)):", dict(Counter(r["review"] for r in SYN)), "— 학습은", repr(SYN_REVIEW_OK), "·", repr(SYN_FACT_OK), "· 목표(측정용 171)는", repr(SYN_REVIEW_OK), "만")
assert all(r["review"] == SYN_REVIEW_OK and not r["amb_mark"] for r in SYN_TRAIN + SYN_DEV), "검수 「적법」 아닌 행이 학습 · 목표에 있다 (D-325)"
assert all(r["review"] == SYN_FACT_OK and not r["amb_mark"] for r in SYN_TRAIN_FACT), "사실 전제 학습에 다른 낱말 · 애매 표시 행이 있다"
print("측정용으로 뗀 제품:", {c: sorted(p for cc, p in SYN_HOLD if cc == c) for c in SYN_CATS})

rows = list(BASE_TRAIN) + list(SYN_TRAIN) + list(SYN_TRAIN_FACT)      # 🆕 D4-10 — 사실 전제 211 을 더한다
if SMOKE:
    rows = list(BASE_TRAIN)
    random.Random(0).shuffle(rows)
    rows = rows[:300] + SYN_TRAIN[:60] + SYN_TRAIN_FACT[:40]
TRAIN_ROWS = rows
_base_n = len({(norm_text(r["text"]), is_reason(r)) for r in pool0})      # D4-00 의 병합 규칙이면 몇 행인가
TRAIN_STATS = {"학습 행": len(rows), "골든 행": len(BASE_TRAIN), "D4-00 병합이면 골든 행": _base_n,
               "정상 합성(음성)": sum(r["origin"] == "synthetic_lawful" for r in rows), "그중 사실 전제": sum(r["review"] == SYN_FACT_OK for r in rows if r["origin"] == "synthetic_lawful"), "음성": sum(1 for r in rows if not r["target"]),
               "품목별": {c: sum(1 for r in rows if cat_of(r) == c) for c in CAT_VALUES + [CAT_UNKNOWN]},
               **{l: sum(l in r["target"] for r in rows) for l in LABEL_LIST}}
print("\n■ 학습셋" + ("   (SMOKE — 줄였다)" if SMOKE else ""))
display(pd.Series({k: v for k, v in TRAIN_STATS.items() if k != "품목별"}).to_frame("값").T)
print("품목별 학습 행:", TRAIN_STATS["품목별"])
print("dev 품목별:", dict(Counter(cat_of(r) for r in dev_rows)), "· 그중 유형 있는 위반:", dict(Counter(cat_of(r) for r in dev_rows if r["bucket"] == "scored")))
EXPECT_TRAIN = {"학습 행": 4511, "골든 행": 3966, "D4-00 병합이면 골든 행": 3964, "정상 합성(음성)": 545, "그중 사실 전제": 211,
                "품목별": {"식품": 1113, "건기식": 1380, "화장품": 400, "모름": 1618}}       # D4-04 의 4,300 + 사실 전제 211(식품 78 · 건기식 68 · 화장품 65) · 로컬 대조 10-14
if GOLDEN_SHA in DEV_EXPECT and not SMOKE and USE_WORD_ROWS is False:
    _got = {k: TRAIN_STATS[k] for k in EXPECT_TRAIN}
    if _got != EXPECT_TRAIN:
        raise SystemExit(f"🔴 학습 행 수가 기대와 다르다 — 받은 {_got} · 기대 {EXPECT_TRAIN}. 학습셋 코드를 건드렸는지 본다")
    print("✅ 학습셋이 D4 기록에 적은 수와 같다")
for _ in [0]:
    assert not (set(r["id"] for r in TRAIN_ROWS) & (set(r["id"] for r in dev_rows) | set(r["id"] for r in SYN_EVAL))), "학습 행에 dev · 측정 행이 섞였다"
    assert not ({norm_text(r["text"]) for r in TRAIN_ROWS} & dev_texts), "학습 행에 dev 와 같은 문구가 있다"
    assert all(r["split"] == "train" or r["origin"] == "synthetic_lawful" for r in TRAIN_ROWS), "골든 train 도 정상 합성도 아닌 행이 학습에 있다"
# ══ 🆕 D4-15 — 재분할 학습 950행에 위와 같은 규칙을 건다 (dev 와 같은 문구 제거 · 조건 M 제외 · 유형 없는 위반 제외 · 낱말 제외 · 같은 문구 병합)
G_STEPS = [("재분할 학습", len(GUIDE["train"]))]
gpool = [r for r in GUIDE["train"] if norm_text(r["text"]) not in dev_texts];             G_STEPS.append(("dev 와 같은 문구 제거", len(gpool)))
gpool = [r for r in gpool if r["cond"] != "M"];                                          G_STEPS.append(("조건 M 제외", len(gpool)))
gpool = [r for r in gpool if not (r["cond"] in ("C", "A", "B") and not r["labels"])];    G_STEPS.append(("유형 없는 위반 제외", len(gpool)))
if not USE_WORD_ROWS:
    gpool = [r for r in gpool if r["unit"] != "낱말"];                                   G_STEPS.append(("낱말 행 제외", len(gpool)))
_seen = {(cat_of(r), norm_text(r["text"]), is_reason(r)) for r in BASE_TRAIN}
_gm = {}
for r in gpool:
    k = (cat_of(r), norm_text(r["text"]), is_reason(r))
    assert k not in _seen, "재분할 학습 행이 골든 학습 행과 같은 문구다 — 로컬 점검(0)과 다르다"
    if k in _gm:
        _gm[k]["target"] = sorted(set(_gm[k]["target"]) | set(r["target"]))
    else:
        _gm[k] = dict(r, target=list(r["target"]))
GUIDE_TRAIN = list(_gm.values());                                                       G_STEPS.append(("같은 문구 병합", len(GUIDE_TRAIN)))
print("재분할 학습:", " → ".join(f"{k} {n:,}" for k, n in G_STEPS))
if SMOKE:
    GUIDE_TRAIN = GUIDE_TRAIN[:100]

def stats_of(rows):
    return {"학습 행": len(rows), "골든 행": sum(r["split"] == "train" for r in rows), "재분할 학습 행": sum(r["split"] == "test_sentence" for r in rows),
            "정상 합성(음성)": sum(r["origin"] == "synthetic_lawful" for r in rows),
            "그중 사실 전제": sum(r["review"] == SYN_FACT_OK for r in rows if r["origin"] == "synthetic_lawful"), "음성": sum(1 for r in rows if not r["target"]),
            "품목별": {c: sum(1 for r in rows if cat_of(r) == c) for c in CAT_VALUES + [CAT_UNKNOWN]},
            **{l: sum(l in r["target"] for r in rows) for l in LABEL_LIST}}
TRAIN = {n: list(TRAIN_ROWS) + (list(GUIDE_TRAIN) if cfg["guide"] else []) for n, cfg in RUN_DEFS.items()}
TRAIN_STATS_BY = {n: stats_of(rows) for n, rows in TRAIN.items()}
print("\n■ 실험별 학습셋")
display(pd.DataFrame({n: {k: v for k, v in st.items() if k != "품목별"} for n, st in TRAIN_STATS_BY.items()}).T)
print("품목별:", {n: st["품목별"] for n, st in TRAIN_STATS_BY.items()})
print("재분할 학습 행의 갈래:", dict(Counter(r["cond"] for r in GUIDE_TRAIN)), "· 유형:", {l: sum(l in r["target"] for r in GUIDE_TRAIN) for l in LABEL_LIST})
_guide_ids = set(RESPLIT_IDS["train"])
for n in RUNS:
    assert not (set(r["id"] for r in TRAIN[n]) & (set(r["id"] for r in dev_rows) | set(r["id"] for r in SYN_EVAL))), "학습 행에 dev · 측정 행이 섞였다"
    assert not ({norm_text(r["text"]) for r in TRAIN[n]} & dev_texts), "학습 행에 확장 dev 와 같은 문구가 있다"
    assert all(r["split"] == "train" or r["origin"] == "synthetic_lawful" or r["id"] in _guide_ids for r in TRAIN[n]), "골든 train · 정상 합성 · 재분할 학습 밖의 행이 학습에 있다"
print("✅ 학습 행 ∩ 확장 dev · 측정 행 = 0 · 봉인 1,141행은 메모리에 없다")


## 5. 입력 · 손실 · 지표

- **입력만 다르다.** 문장쌍으로 넣는다 — 앞이 품목, 뒤가 문장. 길면 **문장 쪽만** 자른다.
- 학습 배치를 만들 때마다 품목이 있는 행의 50% 를 「모름」으로 바꿔 넣는다 (dev · 정상 문구 측정은 바꾸지 않는다).
- 손실 · 지표는 D4-00 과 같다 — Focal Loss(γ 1.5) · α = √(음성/양성) 을 1 ~ 5 로 자른 값 · 6종 dev macro PR-AUC 로 회차를 고른다((가) 품목 입력).


In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          EarlyStoppingCallback, DataCollatorWithPadding, set_seed)
from sklearn.metrics import average_precision_score, f1_score

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def multihot(labels):
    v = np.zeros(N_ALL, dtype=np.float32)
    for l in labels:
        v[LABEL2ID[l]] = 1.0
    return v

def encode_pairs(cats, texts, **kw):
    """[CLS] 품목 [SEP] 문장 [SEP] — 길면 문장 쪽만 자른다."""
    return tokenizer(list(cats), list(texts), truncation="only_second", max_length=MAX_LEN, return_token_type_ids=True, **kw)

def to_ds(rows, train):
    """행마다 두 벌을 미리 토크나이즈한다 — 품목을 넣은 것 · 「모름」을 넣은 것. 어느 쪽을 쓸지는 배치를 만들 때 정한다."""
    texts = [r["text"] for r in rows]
    a = encode_pairs([cat_of(r) for r in rows], texts)
    u = encode_pairs([CAT_UNKNOWN] * len(rows), texts)
    return Dataset.from_dict({
        "ids_cat": a["input_ids"], "tt_cat": a["token_type_ids"], "ids_unk": u["input_ids"], "tt_unk": u["token_type_ids"],
        "labels": [multihot(r["target"]).tolist() for r in rows],
        # 「모름」으로 바꿀 확률 — 학습 행 · 품목이 있을 때만. dev 는 0 (바꾸지 않는다)
        "drop": [CAT_DROP if (train and cat_of(r) != CAT_UNKNOWN) else 0.0 for r in rows]})

class CatCollator:
    """배치를 만들 때 행마다 품목 / 「모름」 중 하나를 고른다. 🔴 Baseline 과 다른 곳은 여기다."""
    def __call__(self, feats):
        batch = []
        for f in feats:
            unk = f["drop"] > 0 and random.random() < f["drop"]
            ids, tt = (f["ids_unk"], f["tt_unk"]) if unk else (f["ids_cat"], f["tt_cat"])
            batch.append({"input_ids": ids, "token_type_ids": tt, "attention_mask": [1] * len(ids)})
        out = tokenizer.pad(batch, return_tensors="pt")
        out["labels"] = torch.tensor([f["labels"] for f in feats], dtype=torch.float32)
        return out

collator = CatCollator()

Y_DEV = np.stack([multihot(r["target"]) for r in dev_rows]).astype(int)
dev_ds = to_ds(dev_rows, train=False)          # 회차 선택은 (가) 품목을 넣은 dev 로 한다

def pos_weight_of(rows):
    Y = np.stack([multihot(r["target"]) for r in rows])
    pos = Y.sum(0); neg = len(Y) - pos
    return np.clip(np.sqrt(neg / np.clip(pos, 1e-6, None)), 1, POS_WEIGHT_CAP)

class FocalTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._pw = torch.tensor(pos_weight, dtype=torch.float32)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits.float()
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * labels + (1 - p) * (1 - labels)
        loss = ((self._pw.to(logits.device) * labels + (1 - labels)) * (1 - p_t) ** FOCAL_GAMMA * bce).mean()
        return (loss, outputs) if return_outputs else loss

def safe(fn, y, p):
    return float(fn(y, p)) if 0 < y.sum() < len(y) else None

def macro_prauc(y, p, n=N_SEL):
    # 앞 n 칸의 PR-AUC 평균 — 양성이 있는 유형만. n=6 이 모델을 고르는 기준이다
    v = [safe(average_precision_score, y[:, i], p[:, i]) for i in range(n)]
    v = [x for x in v if x is not None]
    return float(np.mean(v)) if v else float("nan")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    p = 1 / (1 + np.exp(-logits))
    return {"macro_prauc": macro_prauc(labels[SEL_MASK].astype(int), p[SEL_MASK])}

TH_GRID = np.arange(0.1, 0.9001, 0.025)
def pick_thresholds(y, p):
    th = {}
    for i, l in enumerate(CONF8):
        if y[:, i].sum() < 10:
            th[l] = 0.5
            continue
        fs = [f1_score(y[:, i], p[:, i] >= t, zero_division=0) for t in TH_GRID]
        th[l] = float(round(TH_GRID[int(np.argmax(fs))], 3))
    th[WAITING[0]] = TH_OFF
    return th

def predict_probs(mdl, rows, mode, bs=64):
    """mode = 「품목」(골든의 품목 · 없으면 「모름」) 또는 「모름」(전부 「모름」)."""
    assert mode in MODES, mode
    mdl.eval()
    dev_ = next(mdl.parameters()).device
    out = []
    for s in range(0, len(rows), bs):
        part = rows[s:s + bs]
        cats = [cat_of(r) if mode == "품목" else CAT_UNKNOWN for r in part]
        enc = encode_pairs(cats, [r["text"] for r in part], padding=True, return_tensors="pt").to(dev_)
        with torch.no_grad():
            out.append(torch.sigmoid(mdl(**enc).logits.float()).cpu().numpy())
    return np.concatenate(out)

print("dev 양성 (선택 방식", SCORE_VIEW, "):", {l: int(Y_DEV[SEL_MASK, i].sum()) for i, l in enumerate(LABEL_LIST)})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — CPU 로는 매우 느리다. 런타임 유형을 GPU 로 바꾼다")

_ex = next((r for r in dev_rows if cat_of(r) != CAT_UNKNOWN), dev_rows[0])
_e = encode_pairs([cat_of(_ex)], [_ex["text"]])
print("입력 예 —", " ".join(tokenizer.convert_ids_to_tokens(_e["input_ids"][0])[:14]), "…")
_lens = [len(x) for x in to_ds(TRAIN_ROWS[:2000], train=True)["ids_cat"]]
print(f"학습 앞 {len(_lens):,}행의 토큰 길이 — 최대 {max(_lens)} · 상한 {MAX_LEN} 에 닿은 행 {sum(1 for x in _lens if x >= MAX_LEN)}")
from transformers import AutoConfig
assert int(getattr(AutoConfig.from_pretrained(MODEL_NAME), "max_position_embeddings", MAX_LEN)) >= MAX_LEN, "모델이 받는 길이보다 상한이 길다"

## 6. 학습 — 시드 3개

🔒 **D4 기록에 기준과 목표를 먼저 적고 학습한다.** 시드 하나가 끝나면 공통 dev · 정상 문구의 확률을 두 입력((가) 품목 · (나) 모름)으로 적는다 — 다시 실행하면 끝난 시드는 건너뛴다(`♻️`).


In [ ]:
print("🔒 판정 기준은 D4 실험기록 D4-15 에 먼저 적었다 — 이 노트북은 확률만 낸다. 판정은 로컬 d4_15_resplit.py")

CFG_SIG = [NOTEBOOK, GOLDEN_SHA, DEV_SHA, EXT_DEV_SHA, SCORE_VIEW, MODEL_NAME, MAX_EPOCHS, PATIENCE, BATCH, LR, WARMUP, WEIGHT_DECAY, MAX_LEN,
           FOCAL_GAMMA, POS_WEIGHT_CAP, USE_WORD_ROWS, SMOKE, CAT_VALUES, CAT_UNKNOWN, CAT_DROP]
def sig_of(name, seed):
    ids = hashlib.sha256("|".join(f"{r['id']}:{cat_of(r)}" for r in TRAIN[name]).encode()).hexdigest()
    ev = hashlib.sha256("|".join(r["id"] for r in SYN_EVAL).encode()).hexdigest()
    return hashlib.sha256(json.dumps(CFG_SIG + [name, seed, ids, SYN_SHA, ev], ensure_ascii=False).encode()).hexdigest()[:16]

def train_seed(name, seed, get_ds, pw):
    tag = run_tag(name)
    keep, local = f"{OUT_ROOT}/{tag}/seed{seed}", f"./runs/{tag}/seed{seed}"
    done = f"{keep}/done.json"
    _npys = [f"{keep}/{k}_p_{m}.npy" for k in ("dev", "syn") for m in MODES]
    if os.path.exists(done) and all(os.path.exists(x) for x in _npys):
        res = json.load(open(done, encoding="utf-8"))
        if res.get("sig") == sig_of(name, seed):
            for m in MODES:
                res[f"dev_p_{m}"] = np.load(f"{keep}/dev_p_{m}.npy")
                res[f"syn_p_{m}"] = np.load(f"{keep}/syn_p_{m}.npy")
            print(f"♻️ {name} · seed {seed}: 이미 끝남 — 건너뜀 (best epoch {res['best_epoch']})")
            return res
        print(f"{name} · seed {seed}: 저장된 기록이 지금 설정과 다르다 — 다시 학습한다")
    t0 = time.time()
    train_ds = get_ds()
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_ALL, problem_type="multi_label_classification",
        id2label=dict(enumerate(LABEL_LIST)), label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=f"./runs/{tag}/ckpt{seed}", eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, num_train_epochs=MAX_EPOCHS,
        learning_rate=LR, weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * MAX_EPOCHS),
        load_best_model_at_end=True, metric_for_best_model="macro_prauc", greater_is_better=True,
        logging_steps=50, report_to="none", remove_unused_columns=False, seed=seed,
        fp16=torch.cuda.is_available())
    trainer = FocalTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds, data_collator=collator,
                           compute_metrics=compute_metrics, pos_weight=pw,
                           callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    hist = [h for h in trainer.state.log_history if "eval_macro_prauc" in h]
    best = max(hist, key=lambda h: h["eval_macro_prauc"])
    P = {}
    for m in MODES:                                         # (가) 품목 · (나) 모름 — 같은 모델 · 입력만 다르게
        P[f"dev_p_{m}"] = predict_probs(trainer.model, dev_rows, m)
        P[f"syn_p_{m}"] = predict_probs(trainer.model, SYN_EVAL, m)      # 정상 문구(측정용 + 참고) — 학습에 넣지 않은 행
    shutil.rmtree(local, ignore_errors=True)
    trainer.save_model(local)
    shutil.rmtree(f"./runs/{tag}/ckpt{seed}", ignore_errors=True)
    res = {"sig": sig_of(name, seed), "seed": seed, "best_epoch": int(round(best["epoch"])), "epochs_run": int(round(hist[-1]["epoch"])),
           "curve": [[int(round(h["epoch"])), round(h["eval_macro_prauc"], 4)] for h in hist], "minutes": round((time.time() - t0) / 60, 1)}
    os.makedirs(keep, exist_ok=True)
    for k, v in P.items():
        np.save(f"{keep}/{k}.npy", v)
    shutil.copy(f"{local}/config.json", f"{keep}/config.json")
    json.dump(res, open(done, "w", encoding="utf-8"), ensure_ascii=False)       # 🔴 done.json 을 맨 끝에 쓴다 — 있으면 이 시드는 끝난 것이다
    res.update(P)
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

def scheme_of(name, R):
    s, st = R["best_seed"], TRAIN_STATS_BY[name]
    return {
        "label_list": LABEL_LIST, "label_thresholds": R["thresholds"],
        "confirmed_labels": CONF8, "waiting_labels": WAITING, "legacy_confirmed_labels": SEL,
        "experiment_only": True, "experiment_note": "🔴 입력이 문장쌍(품목 · 문장)이다 — 앱 로더(문장만)에 올리지 않는다(D-336 집행 대기). 그래서 label_scheme.experiment.json 이다",
        "input_format": {"pair": True, "first": "품목", "values": CAT_VALUES, "unknown": CAT_UNKNOWN, "truncation": "only_second", "max_len": MAX_LEN, "cat_drop": CAT_DROP},
        "backbone": MODEL_NAME, "notebook": f"{NOTEBOOK} (박수진 · D4-10 + 재분할 v1 학습 — D-347)", "experiment": run_tag(name),
        "baseline": "D4-10 = copylane-encoder-kcbert-D4-10-품목입력사실전제 (채택 · ef2c12a3b90f) · 같은 확장 dev 비교는 D4-10재현-확장dev (시드 48 · 49 · 50)",
        "synthetic_lawful_rows": st["정상 합성(음성)"], "synthetic_lawful_sha256": SYN_SHA, "synthetic_lawful_reviews": [SYN_REVIEW_OK, SYN_FACT_OK], "synthetic_lawful_fact_rows": st["그중 사실 전제"],
        "target_rule": "encoder_target — 조건 M · D · L 음성 · 전제 건기식_비인정 거짓_과장 → 건강기능식품_오인 (D-326 보충 12)",
        "seed": s, "best_epoch": R["seeds"][s]["best_epoch"], "golden_sha256": GOLDEN_SHA, "golden_rows": list(EXPECTED_ROWS),
        "train_rows": st["학습 행"], "train_by_cat": st["품목별"], "train_label_pos": {l: st[l] for l in LABEL_LIST},
        "dev_sha": DEV_SHA, "dev_rows": len(DEV749), "score_view": SCORE_VIEW,
        # 🆕 D4-15 (D-347 결정 4) — 평가 도구가 --dev --guide-dev 로 이 지문을 대조한다 · dev_sha 는 devsplit 것 그대로
        "resplit": "v1", "resplit_train_sha": RESPLIT_EXPECT["train"][1], "resplit_train_rows": RESPLIT_EXPECT["train"][0],
        "resplit_dev_sha": RESPLIT_EXPECT["dev"][1], "resplit_dev_rows": RESPLIT_EXPECT["dev"][0],
        "resplit_train_used_rows": st["재분할 학습 행"], "dev_rows_extended": len(dev_rows), "dev_sha_extended": EXT_DEV_SHA,
        "threshold_note": "미리 보기 문턱 — (가) 품목 입력 · 가장 좋은 시드의 확장 dev F1 최대. 판정 문턱은 d4_15_resplit.py 가 입력별(품목 · 모름) D-324 로 다시 고른다",
        "dev_macro_prauc6_by_seed": {str(k): v["prauc6"] for k, v in R["seeds"].items()},
        "final_test": False, "test_note": "이 노트북은 test 를 채점하지 않는다 (D-175)", "smoke": SMOKE,
    }

def export_run(name, R):
    """시드 · 입력별 dev 행별 확률 CSV · 정상 문구 행별 확률 CSV · 문턱 zip · (스위치) 가장 좋은 시드의 모델 zip."""
    tag, s = run_tag(name), R["best_seed"]
    scheme, out = scheme_of(name, R), []
    for sd in SEEDS:
        for m in MODES:
            csv_path = f"{OUT_ROOT}/{tag}_{m}_seed{sd}_dev행별확률.csv"
            with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:      # 문구는 넣지 않는다 (id · 갈래 · 확률만)
                w = csv.writer(f)
                w.writerow(["id", "조건", "bucket", "이유구역", "품목입력"] + [f"p_{l}" for l in LABEL_LIST])
                for r, row in zip(dev_rows, R["seeds"][sd][f"dev_p_{m}"]):
                    w.writerow([r["id"], r["cond"] or "", r["bucket"], int(is_reason(r)), cat_of(r) if m == "품목" else CAT_UNKNOWN] + [f"{x:.6f}" for x in row])
            out.append(os.path.basename(csv_path))
            csv_path = f"{OUT_ROOT}/{tag}_{m}_seed{sd}_합성정상_행별확률.csv"
            with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:
                w = csv.writer(f)
                w.writerow(["id", "품목", "갈래"] + [f"p_{l}" for l in LABEL_LIST])
                for r, row in zip(SYN_EVAL, R["seeds"][sd][f"syn_p_{m}"]):
                    w.writerow([r["id"], r["cat"], ("애매" if r["amb_mark"] else f"참고-{r['review']}") if r["amb"] else "합성정상dev"] + [f"{x:.6f}" for x in row])
            out.append(os.path.basename(csv_path))
    with zipfile.ZipFile(f"{OUT_ROOT}/{tag}_scheme.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("label_scheme.experiment.json", json.dumps(scheme, ensure_ascii=False, indent=2))
        z.write(f"{OUT_ROOT}/{tag}/seed{s}/config.json", "config.json")
    out.append(f"{tag}_scheme.zip")
    local = f"./runs/{tag}/seed{s}"
    if SAVE_MODEL_ZIP and name == RUN and os.path.exists(f"{local}/config.json"):
        sdir = f"./export/{tag}"
        shutil.rmtree(sdir, ignore_errors=True)
        mdl = AutoModelForSequenceClassification.from_pretrained(local)
        assert [mdl.config.id2label[i] for i in range(mdl.config.num_labels)] == LABEL_LIST, "저장 칸 순서가 다르다 (D-220)"
        mdl.save_pretrained(sdir, safe_serialization=True)
        tokenizer.save_pretrained(sdir)
        json.dump(scheme, open(f"{sdir}/label_scheme.experiment.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)   # 🔴 label_scheme.json 이 아니다
        zp = shutil.make_archive(f"./export/{tag}", "zip", sdir)
        shutil.copy(zp, f"{OUT_ROOT}/{tag}.zip")
        out.append(f"{tag}.zip")
        os.remove(zp); shutil.rmtree(sdir, ignore_errors=True)
        del mdl
    return out

RESULTS = {}
T_ALL = time.time()
for name in RUNS:
    tag, rows = run_tag(name), TRAIN[name]
    pw = pos_weight_of(rows)
    print(f"\n{'=' * 22} {name} · 학습 {len(rows):,}행 {'=' * 22}")
    print("pos_weight:", {l: round(float(pw[i]), 2) for i, l in enumerate(LABEL_LIST)})
    _cache, seeds = {}, {}
    def get_ds():                                 # 학습할 시드가 있을 때만 토크나이즈한다
        if "ds" not in _cache:
            _cache["ds"] = to_ds(rows, train=True)
        return _cache["ds"]
    for seed in SEEDS:
        res = train_seed(name, seed, get_ds, pw)
        y = Y_DEV[SEL_MASK]
        for m in MODES:
            p = res[f"dev_p_{m}"][SEL_MASK]
            res[f"prauc6_{m}"] = macro_prauc(y, p, N_SEL)
        res["prauc6"] = res["prauc6_품목"]
        res["thresholds"] = pick_thresholds(y, res["dev_p_품목"][SEL_MASK])
        seeds[seed] = res
        print(f"  seed {seed}: 6종 dev macro PR-AUC (가) 품목 {res['prauc6_품목']:.4f} · (나) 모름 {res['prauc6_모름']:.4f} · best epoch {res['best_epoch']}/{res['epochs_run']} · 곡선 {res['curve']} · {res['minutes']}분")
    best = max(SEEDS, key=lambda s: seeds[s]["prauc6"])
    R = {"seeds": seeds, "best_seed": best, "thresholds": seeds[best]["thresholds"],
         "prauc6_mean": float(np.mean([seeds[s]["prauc6"] for s in SEEDS])), "prauc6_std": float(np.std([seeds[s]["prauc6"] for s in SEEDS]))}
    RESULTS[name] = R
    print(f"[{name}] (가) 품목 6종 dev macro PR-AUC {R['prauc6_mean']:.4f} ± {R['prauc6_std']:.4f} · 가장 좋은 시드 {best}")
    print("  저장:", export_run(name, R))
    for s in SEEDS:
        if s != best:
            shutil.rmtree(f"./runs/{tag}/seed{s}", ignore_errors=True)
print(f"\n학습 끝 — {round((time.time() - T_ALL) / 60, 1)}분")


## 7. 결과 — 미리 보기 (확장 dev)

입력별((가) 품목 · (나) 모름) · 시드별 6종 PR-AUC 와 정상 문구에 후보가 서는 행을 F1 최대 문턱((가) 기준)으로 미리 본다.
🔴 **판정은 이 표가 아니다** — D4-00 과 같은 D-324 문턱 · 같은 관문으로 로컬 스크립트가 잰다.


In [ ]:
SYN_IS_DEV = np.array([not r["amb"] for r in SYN_EVAL])
_fd = np.array([r["review"] == SYN_FACT_OK and not r["amb_mark"] for r in SYN_EVAL])
_g = np.array([i >= len(DEV749) for i in range(len(dev_rows))])
sc = B == "scored"
rows_ = []
for name in RUNS:
    R = RESULTS[name]
    for s in SEEDS:
        th = np.array([R["seeds"][s]["thresholds"][l] for l in CONF8])
        for m in MODES:
            fl = R["seeds"][s][f"dev_p_{m}"][:, :N8] >= th
            q = R["seeds"][s][f"syn_p_{m}"][:, :N8] / th
            rows_.append({"실험": name, "시드": s, "입력": m, "6종 PR-AUC (확장 dev)": round(R["seeds"][s][f"prauc6_{m}"], 4),
                          "위반 탐지 공통 · 재분할": f"{int(fl[sc & ~_g].any(1).sum())}/{int((sc & ~_g).sum())} · {int(fl[sc & _g].any(1).sum())}/{int((sc & _g).sum())}",
                          "정상 문구 후보": f"{int((q[SYN_IS_DEV] >= 1).any(1).sum())}/{int(SYN_IS_DEV.sum())}",
                          "사실 전제 후보": f"{int((q[_fd] >= 1).any(1).sum())}/{int(_fd.sum())}"})
display(pd.DataFrame(rows_))
print("🔴 미리 보기다 — F1 최대 문턱. 판정은 로컬 d4_15_resplit.py (D-324 · 목표 = D4-15a 기준선 재현)")
if SMOKE:
    print("⚠️ SMOKE 실행 — 위 수치는 의미 없다")

SUMMARY = {"notebook": NOTEBOOK, "golden_sha256": GOLDEN_SHA, "dev_sha": DEV_SHA, "dev_rows": len(DEV749), "dev_rows_extended": len(dev_rows), "dev_sha_extended": EXT_DEV_SHA,
           "resplit": {k: list(v) for k, v in RESPLIT_EXPECT.items()}, "smoke": SMOKE, "train": TRAIN_STATS_BY, "guide_steps": G_STEPS,
           "cat_drop": CAT_DROP, "max_len": MAX_LEN, "syn_sha256": SYN_SHA,
           "runs": {n: {"best_seed": RESULTS[n]["best_seed"], "seeds": {str(s): {k: RESULTS[n]["seeds"][s][k] for k in ("prauc6_품목", "prauc6_모름", "best_epoch", "epochs_run", "curve", "minutes", "thresholds")} for s in SEEDS}} for n in RUNS}}
json.dump(SUMMARY, open(f"{OUT_ROOT}/results_D4-15.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)

_csvs = [f"{run_tag(n)}_{m}_seed{s}_{k}.csv" for n in RUNS for m in MODES for s in SEEDS for k in ("dev행별확률", "합성정상_행별확률")]
_miss = [f for f in _csvs if not os.path.exists(f"{OUT_ROOT}/{f}")]
if _miss:
    print("🟡 CSV 가 덜 나왔다:", _miss)
else:
    with zipfile.ZipFile(f"{OUT_ROOT}/D4-15_csv.zip", "w", zipfile.ZIP_DEFLATED) as _z:
        for f in _csvs + ["results_D4-15.json"]:
            _z.write(f"{OUT_ROOT}/{f}", f)
    print(f"\n📦 받을 파일: {os.path.abspath(OUT_ROOT)}/D4-15_csv.zip ({len(_csvs)}개 + 요약) — PC 의 Downloads\\copylane_D4\\D4-15_csv\\ 에 푼다")
    print(f"📦 모델: {OUT_ROOT}/{run_tag(RUN)}.zip — PC 의 Downloads\\copylane_D4\\D4-15_model\\")


## 8. 다음 단계

**1) 받을 파일** — `D4-15_csv.zip` → `Downloads\copylane_D4\D4-15_csv\` · 모델 zip → `Downloads\copylane_D4\D4-15_model\` · D4-15a 노트북의 `D4-15a_csv.zip` → `Downloads\copylane_D4\D4-15a_csv\`

**2) 판정 (로컬)**

```
uv run python docs\psj\e2e_prototype\d4_15_resplit.py --base-dir <D4-15a_csv> --dir <D4-15_csv> --review-csv <검수판 CSV> --out-md docs\psj\reports\D4-15_재분할학습_dev_YYYYMMDD.md
```

**3) 배포 폴더** — `make_d415_scheme.py` 가 D-324 문턱 두 벌 + 재분할 지문으로 `label_scheme.json` 을 쓰고 zip 을 만든다.

**4) 올리지 않는 것** — 모델 zip · 행별 확률 CSV · 재분할 목록 txt · 실행 출력 · `golden.jsonl` · 정상 문구 CSV (D-249). 노트북은 출력을 지우고 올린다.
